# 📱 Liquidity Stress Prediction Challenge - Starter Notebook

Welcome to the my notebook for this challenge.

In this challenge, my goal is to build a machine learning model that predicts whether a customer is likely to experience liquidity stress, using historical mobile money behaviour.

This notebook walks through a simple end-to-end pipeline:

- Loading and exploring the data
- Preprocessing
- Training a baseline model
- Evaluating performance
- Generating predictions for submission

The baseline model used here is **Logistic Regression**.


In [28]:
# 📚 Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)


In [29]:
# 📥 Load the Data
train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")

# Peek at the data
train.head()


,arpu,age,gender,region,smartphone,segment,earning_pattern,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_volume,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_highest_amount,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_highest_amount,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_highest_amount,m3_received_senders,m4_received_volume,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_volume,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_volume,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_highest_amount,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_highest_amount,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_highest_amount,m3_deposit_agents,m4_deposit_volume,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_volume,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_volume,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_highest_amount,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_highest_amount,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_highest_amount,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_highest_amount,m4_withdraw_agents,m5_withdraw_volume,m5_withdraw_total_value,m5_withdraw_highe

## Data cleaning and exploration

Here we inspect the data for important information that could affect the training process and model building.

In [30]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Columns: 184 entries, arpu to ID
dtypes: float64(134), int64(44), str(6)
memory usage: 56.2 MB


In [31]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Columns: 183 entries, arpu to ID
dtypes: float64(134), int64(43), str(6)
memory usage: 41.9 MB


In [32]:
# Check missing values
missing_summary = train.isnull().sum().sort_values(ascending=False)
# missing_summary[missing_summary > 0].head(20)
missing_summary ## There are no missing values

arpu                         0
age                          0
gender                       0
region                       0
smartphone                   0
                            ..
m4_daily_avg_bal             0
m5_daily_avg_bal             0
m6_daily_avg_bal             0
liquidity_stress_next_30d    0
ID                           0
Length: 184, dtype: int64

In [33]:
missing_summary[missing_summary > 0]

Series([], dtype: int64)

There are no missing values.

### 🏷️ Encoding Categorical Variables

We use **one-hot encoding** to convert categorical columns into numeric format so they can be used by the model.


In [34]:
# Identify the ID column, target column and feature columns
TARGET = "liquidity_stress_next_30d"
ID_COL = "ID"
object_type=['str','object']

feature_cols = [c for c in train.columns if c not in [ID_COL, TARGET]]
cat_cols = [c for c in feature_cols if train[c].dtype in object_type]
num_cols = (
    train.select_dtypes(include=['number'])
    .drop(columns=[TARGET], errors='ignore')
    .columns.tolist()
)

cat_cols


['gender', 'region', 'smartphone', 'segment', 'earning_pattern']

In [35]:
## Inspect categorical columns for misspelling
for col in cat_cols:
  print(f"Unique values in {col}:", train[col].unique())

Unique values in gender: <StringArray>
['F', 'M']
Length: 2, dtype: str
Unique values in region: <StringArray>
[              'Central Zone',         'Northern Highlands',
      'Lake Zone (Northwest)', 'Coastal & Eastern Corridor',
         'Southern Highlands',                   'Islander',
               'Western Zone']
Length: 7, dtype: str
Unique values in smartphone: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Unique values in segment: <StringArray>
['MVC', 'LVC', 'HVC']
Length: 3, dtype: str
Unique values in earning_pattern: <StringArray>
['Monthly Earner', 'Weekly Earner', 'Irregular Earner']
Length: 3, dtype: str


In [36]:
for col in cat_cols:
  print(f"Unique values in {col}:", test[col].unique())

Unique values in gender: <StringArray>
['F', 'M']
Length: 2, dtype: str
Unique values in region: <StringArray>
[              'Central Zone',         'Northern Highlands',
      'Lake Zone (Northwest)', 'Coastal & Eastern Corridor',
         'Southern Highlands',                   'Islander',
               'Western Zone']
Length: 7, dtype: str
Unique values in smartphone: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Unique values in segment: <StringArray>
['MVC', 'LVC', 'HVC']
Length: 3, dtype: str
Unique values in earning_pattern: <StringArray>
['Monthly Earner', 'Weekly Earner', 'Irregular Earner']
Length: 3, dtype: str


In [37]:
## Split to Features and Target
X=train[feature_cols]
y=train[TARGET]

Split the training data to training and validation. We apply statify to ensure that equal proportion of the 1s and 0s are accounted for within each split

In [38]:

# ----------------------------- SPLIT 60 / 15 / 15 / 10 -----------                 
X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.4,random_state=42,stratify=y)# 60% | 40%


We will now apply One Hot Encoding for the the categorical columns in the X_train and X_val

In [39]:
# 1. Initialize the encoder (drop='first' mimics what you used before)
ohe = OneHotEncoder(drop='first', sparse_output=False)

# 2. Fit and transform ONLY on the categorical training columns
train_encoded_array = ohe.fit_transform(X_train[cat_cols])
# 3. Transform ONLY on the categorical validation columns
val_encoded_array = ohe.transform(X_val[cat_cols])

# 4. Get the new feature names generated by OHE
encoded_feature_names = ohe.get_feature_names_out(cat_cols)

# 4. Convert the NumPy array back into a pandas DataFrame
X_train_encoded = pd.DataFrame(
    train_encoded_array,
    columns=encoded_feature_names,
    index=X_train.index
)

X_val_encoded = pd.DataFrame(
    val_encoded_array,
    columns=encoded_feature_names,
    index=X_val.index
)

X_train_encoded.head()


,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
29002,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
12145,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0
15100,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
21225,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
9189,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


In [40]:
# 3. Drop the old unencoded categorical columns from X_train and X_val
# and concatenate the new OHE columns alongside the numeric ones
X_train_ohe = pd.concat(
    [X_train.drop(columns=cat_cols), X_train_encoded],
    axis=1
)

X_val_ohe = pd.concat(
    [X_val.drop(columns=cat_cols), X_val_encoded],
    axis=1
)

X_train_ohe.head()

,arpu,age,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_volume,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_highest_amount,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_highest_amount,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_highest_amount,m3_received_senders,m4_received_volume,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_volume,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_volume,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_highest_amount,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_highest_amount,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_highest_amount,m3_deposit_agents,m4_deposit_volume,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_volume,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_volume,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_highest_amount,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_highest_amount,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_highest_amount,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_highest_amount,m4_withdraw_agents,m5_withdraw_volume,m5_withdraw_total_value,m5_withdraw_highest_amount,m5_withdraw_agents,m6_withdraw_volume,m

### Inspect for Feature Importance to the target variable using the correlation
We will apply correlation to see the linear relationship between each feature and the target

In [41]:
# 1. Calculate correlation with y_train and convert the resulting Series to a DataFrame
correlation_df = X_train_ohe.corrwith(y_train).to_frame(name='Correlation')

# 2. Sort by absolute correlation to see the most impactful features at the top
correlation_df = (
    correlation_df
    .assign(Abs_Correlation=correlation_df['Correlation'].abs())
    .sort_values(by='Abs_Correlation', ascending=False)
    .drop(columns=['Abs_Correlation'])
)

# Display the DataFrame
correlation_df

,Correlation
m6_daily_avg_bal,0.078363
m5_daily_avg_bal,0.077523
m4_daily_avg_bal,0.075849
m1_deposit_agents,-0.075795
m2_deposit_agents,-0.074858
...,...
m5_paybill_volume,0.001024
smartphone_Yes,-0.000996
region_Lake Zone (Northwest),0.000891
region_Northern Highlands,0.000511


The Linear relationships are very weak this could mean the relationships are non-linear

### CHECK FOR COLLINEARITY (within the features)
We will check for collinearity amongst the numerical variables.

In [42]:
## Correlation function that find the linear relationship between features using a given threshold to identify collinear relationship

def correlation(dataset, threshold):
    col_corr = set()
    col_pair = set()

    corr_matrix = dataset.corr()

    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > threshold:
                colname1 = corr_matrix.columns[i]
                colname2 = corr_matrix.columns[j]

                col_pair.add((colname1, colname2, corr_matrix.iloc[i, j]))
                col_corr.add(colname1)

    return col_corr, col_pair

In [43]:
col_corr, col_pair = correlation(X_train_ohe[num_cols], 0.7)
col_pair

{('m1_daily_avg_bal', 'arpu', np.float64(0.7313172718712474)),
 ('m1_deposit_highest_amount',
  'm1_deposit_volume',
  np.float64(0.7160475734203413)),
 ('m1_merchantpay_highest_amount',
  'm1_merchantpay_total_value',
  np.float64(0.7433401027397554)),
 ('m1_merchantpay_highest_amount',
  'm1_merchantpay_volume',
  np.float64(0.7220381763773687)),
 ('m1_merchantpay_total_value',
  'm1_merchantpay_volume',
  np.float64(0.7172495593424918)),
 ('m1_mm_send_highest_amount', 'arpu', np.float64(0.7306259535307245)),
 ('m1_mm_send_highest_amount',
  'm1_mm_send_volume',
  np.float64(0.7227908866683658)),
 ('m1_paybill_highest_amount',
  'm1_paybill_volume',
  np.float64(0.7246036511778355)),
 ('m1_received_highest_amount', 'arpu', np.float64(0.7191494246133681)),
 ('m1_received_highest_amount',
  'm1_received_volume',
  np.float64(0.7275986583392619)),
 ('m1_transfer_from_bank_highest_amount',
  'm1_transfer_from_bank_volume',
  np.float64(0.7786805200213568)),
 ('m1_withdraw_highest_amount'

These are the collinear pairs. We will have to drop some of the pairs systematically in a way that we compare with the relationship with the target. 

In [44]:
threshold = 0.7

# Strength of each feature's relationship with the target
target_corr = X_train_ohe[num_cols].corrwith(y_train).abs()

# Pairwise feature correlations (upper triangle only, so each pair appears once)
corr = X_train_ohe[num_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

pairs = (
    upper.stack()
    .rename("pair_corr")
    .reset_index()
    .rename(columns={"level_0": "feat_a", "level_1": "feat_b"})
)
pairs = pairs[pairs["pair_corr"] > threshold].sort_values("pair_corr", ascending=False)

# For each pair, drop the feature with the weaker target correlation
to_drop, log = set(), []
for _, r in pairs.iterrows():
    a, b = r["feat_a"], r["feat_b"]
    if a in to_drop or b in to_drop:      # pair already resolved by an earlier drop
        continue
    drop, keep = (a, b) if target_corr[a] < target_corr[b] else (b, a)
    to_drop.add(drop)
    log.append({
        "dropped": drop,
        "kept": keep,
        "pair_corr": round(r["pair_corr"], 3),
        "dropped_target_corr": round(target_corr[drop], 4),
        "kept_target_corr": round(target_corr[keep], 4),
        "gap": round(target_corr[keep] - target_corr[drop], 4),
    })

drop_log = pd.DataFrame(log)

# Pairs where the two features are nearly tied on target correlation: review by hand
close_calls = drop_log[drop_log["gap"] < 0.01]

drop_log

,dropped,kept,pair_corr,dropped_target_corr,kept_target_corr,gap
0,m1_transfer_from_bank_volume,m1_transfer_from_bank_highest_amount,0.779,0.0099,0.0149,0.0050
1,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_volume,0.778,0.0057,0.0260,0.0202
2,m2_transfer_from_bank_volume,m2_transfer_from_bank_highest_amount,0.775,0.0098,0.0147,0.0048
3,m6_transfer_from_bank_volume,m6_transfer_from_bank_highest_amount,0.774,0.0062,0.0152,0.0089
4,m5_transfer_from_bank_volume,m5_transfer_from_bank_highest_amount,0.773,0.0166,0.0185,0.0019
5,m4_transfer_from_bank_volume,m4_transfer_from_bank_highest_amount,0.771,0.0165,0.0253,0.0089
6,m2_merchantpay_highest_amount,m2_merchantpay_total_value,0.747,0.0270,0.0286,0.0016
7,m1_merchantpay_total_value,m1_merchantpay_highest_amount,0.743,0.0268,0.0276,0.0008
8,m2_daily_avg_bal,arpu,0.742,0.0074,0.0286,0.0212
9,m4_merchantpay_highest_amount,m4_merchantpay_total_value,0.742,0.0125,0.0192,0.0067


In [45]:
## number of columns to drop
len(to_drop)

50

In [46]:
## convert to list
features_to_drop=list(to_drop)
features_to_drop

['m5_received_volume',
 'm5_withdraw_volume',
 'm3_transfer_from_bank_highest_amount',
 'm2_paybill_volume',
 'm1_received_highest_amount',
 'm5_merchantpay_total_value',
 'm6_paybill_volume',
 'm2_daily_avg_bal',
 'm5_mm_send_volume',
 'm2_mm_send_volume',
 'm2_merchantpay_volume',
 'm6_merchantpay_volume',
 'm1_deposit_highest_amount',
 'm2_deposit_highest_amount',
 'm5_paybill_volume',
 'm1_merchantpay_volume',
 'm5_merchantpay_volume',
 'm6_merchantpay_highest_amount',
 'm4_merchantpay_highest_amount',
 'm2_transfer_from_bank_volume',
 'm2_merchantpay_highest_amount',
 'm4_merchantpay_volume',
 'm5_deposit_volume',
 'm4_mm_send_volume',
 'm2_received_highest_amount',
 'm1_withdraw_highest_amount',
 'm4_received_volume',
 'arpu',
 'm6_received_volume',
 'm6_withdraw_highest_amount',
 'm1_merchantpay_total_value',
 'm3_paybill_volume',
 'm3_merchantpay_volume',
 'm3_received_highest_amount',
 'm3_merchantpay_highest_amount',
 'm5_transfer_from_bank_volume',
 'm4_paybill_volume',
 'm3

In [47]:
X_train_ohe_adj=X_train_ohe.drop(columns=features_to_drop)
X_train_ohe_adj.head()

,age,x_90_d_activity_rate,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_total_value,m2_merchantpay_merchants,m3_merchantpay_total_value,m3_merchantpay_merchants,m4_merchantpay_total_value,m4_merchantpay_merchants,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_total_value,m6_merchantpay_merchants,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_banks,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_recipients,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_recipients,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_senders,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_agents,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_agents,m5_withdraw_total_value,m5_withdraw_highest_amount,m5_withdraw_agents,m6_withdraw_volume,m6_withdraw_total_value,m6_withdraw_agents,m1_daily_avg_bal,m3_daily_avg_bal,m4_daily_avg_bal,m5_daily_avg_bal,m6_daily_avg_bal,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
29002,62,0.3200,14460.46,4468.75,8.0,0.00,0.00,0.0,12434.87,3570.36,5.0,19690.79,5664.73,4.0,0.00,0.00,0.0,10165.75,7190.36,5.0,1725.72,2.0,11341.27,8.0,0.00,0.0,11726.24,7.0,0.00,0.0,12330.10,14.0,0.00,0.00,0.0,14592.08,11787.13,2.0,0,0.00,0.0,20022.19,14324.13,2.0,0.00,0.00,0.0,30997.07,17643.95,15.0,0,0.00,0.0,143660.35,7487.67,17.0,0,0.00,0.0,0.00,0.00,0.0,24653.09,9742.70,5.0,82513.68,5222.53,18.0,17,76746.74,17.0,18,74602.85,3.0,15,72677.08,15.0,92128.20,8560.97,5.0,0.00,0.00,0.0,107897.25,11111.57,10.0,7,48251.76,5.0,8,188839.73,1.0,0,0.00,0.0,0.00,0.00,0.0,172686.44,14541.22,2.0,0.00,0.00,0.0,6,53732.80,1.0,3,20346.63,1.0,0,0.00,0.0,5,9765.13,1.0,36467.58,4380.47,5.0,4,14500.71,5.0,6245.69,6221.92,10119.83,5591.35,110

In [48]:
X_val_ohe_adj=X_val_ohe.drop(columns=features_to_drop)


In [49]:
X_val_ohe_adj.shape, X_train_ohe_adj.shape

((16000, 139), (24000, 139))

In [50]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score, precision_recall_curve,
)

In [52]:
SEED = 42
ERROR_WEIGHT = 3.0 

X_tr=X_train_ohe_adj.copy()
X_rest=X_val_ohe_adj.copy()
y_tr=y_train.copy()
y_rest=y_val.copy()

X_a, X_rest, y_a, y_rest = train_test_split(
    X_rest, y_rest, test_size=0.625, stratify=y_rest, random_state=SEED) # 15% | 25%
X_b, X_final, y_b, y_final = train_test_split(
    X_rest, y_rest, test_size=0.40, stratify=y_rest, random_state=SEED)  # 15% | 10%

print(f"train={len(y_tr)}  val_A={len(y_a)}  val_B={len(y_b)}  final={len(y_final)}")
print("positive rate -> train: %.3f | A: %.3f | B: %.3f | final: %.3f"
      % (y_tr.mean(), y_a.mean(), y_b.mean(), y_final.mean()))

# ----------------------------- HELPERS ----------------------------
def make_model():
    return lgb.LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )

def best_threshold(y_true, proba):
    """Threshold that maximizes F1."""
    p, r, t = precision_recall_curve(y_true, proba)
    f1 = 2 * p * r / (p + r + 1e-12)
    return t[np.argmax(f1[:-1])]

def evaluate(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "f1": f1_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred),
        "accuracy": accuracy_score(y_true, pred),
        "threshold": float(thr),
    }

# ----------------------------- ITERATIVE MINING -------------------
def run_iterative(mode):
    """
    mode = "hard_only": add only misclassified rows from each val chunk
    mode = "weighted" : add the whole val chunk, misclassified rows get ERROR_WEIGHT
    """
    X_cur, y_cur = X_tr.copy(), y_tr.copy()
    w_cur = np.ones(len(y_cur))
    log, thr = [], None

    for name, Xv, yv in [("val_A", X_a, y_a), ("val_B", X_b, y_b)]:
        model = make_model().fit(X_cur, y_cur, sample_weight=w_cur)
        proba = model.predict_proba(Xv)[:, 1]
        thr = best_threshold(yv, proba)              # threshold picked on this val chunk
        wrong = (proba >= thr).astype(int) != yv

        log.append({"stage": f"{mode}:{name}", "n_train": len(y_cur),
                    "n_wrong": int(wrong.sum()), **evaluate(yv, proba, thr)})

        if mode == "hard_only":
            X_add, y_add = Xv[wrong], yv[wrong]
            w_add = np.ones(int(wrong.sum()))
        else:
            X_add, y_add = Xv, yv
            w_add = np.where(wrong, ERROR_WEIGHT, 1.0)

        X_cur = pd.concat([X_cur, X_add])
        y_cur = np.concatenate([y_cur, y_add])
        w_cur = np.concatenate([w_cur, w_add])

    # Final model, evaluated ONCE on the untouched 10%
    final_model = make_model().fit(X_cur, y_cur, sample_weight=w_cur)
    proba_final = final_model.predict_proba(X_final)[:, 1]
    final_res = {"stage": f"{mode}:FINAL", "n_train": len(y_cur),
                 **evaluate(y_final, proba_final, thr)}
    return log, final_res

# ----------------------------- BASELINE ---------------------------
# Train on all 90% (60+15+15), no mining.
X_90 = pd.concat([X_tr, X_a, X_b])
y_90 = np.concatenate([y_tr, y_a, y_b])

# No clean chunk left to pick a threshold, so use out-of-fold predictions on the 90%
oof = cross_val_predict(
    make_model(), X_90, y_90,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    method="predict_proba")[:, 1]
thr_base = best_threshold(y_90, oof)

base_model = make_model().fit(X_90, y_90)
base_final = {"stage": "baseline:FINAL", "n_train": len(y_90),
              **evaluate(y_final, base_model.predict_proba(X_final)[:, 1], thr_base)}

# ----------------------------- RUN ALL ----------------------------
log_hard, final_hard = run_iterative("hard_only")
log_wtd, final_wtd = run_iterative("weighted")

# Round-by-round log (are the val scores improving from A to B?)
print("\n=== Intermediate stages ===")
print(pd.DataFrame(log_hard + log_wtd).round(4).to_string(index=False))

# Final comparison on the untouched 10%
print("\n=== FINAL 10% comparison ===")
final_table = pd.DataFrame([base_final, final_hard, final_wtd]).set_index("stage")
print(final_table.round(4).to_string())

train=24000  val_A=6000  val_B=6000  final=4000
positive rate -> train: 0.150 | A: 0.150 | B: 0.150 | final: 0.150

=== Intermediate stages ===
          stage  n_train  n_wrong  roc_auc  pr_auc     f1  precision  recall  accuracy  threshold
hard_only:val_A    24000      731   0.8702  0.6325 0.5777     0.6017  0.5556    0.8782     0.2723
hard_only:val_B    24731      697   0.8815  0.6430 0.6019     0.6193  0.5856    0.8838     0.2915
 weighted:val_A    24000      731   0.8702  0.6325 0.5777     0.6017  0.5556    0.8782     0.2723
 weighted:val_B    30000      709   0.8813  0.6429 0.5942     0.6128  0.5767    0.8818     0.2924

=== FINAL 10% comparison ===
                 n_train  roc_auc  pr_auc      f1  precision  recall  accuracy  threshold
stage                                                                                    
baseline:FINAL     36000   0.8629  0.5877  0.5407     0.5014  0.5867    0.8505     0.2464
hard_only:FINAL    25428   0.8538  0.5708  0.5283     0.5283  0.52

## Create submission 

In [53]:
SUBMISSION_TARGET = "Target"

# Create a sample submission template directly from the test IDs
submission = pd.DataFrame({
    "ID": test["ID"],
    SUBMISSION_TARGET: 0.5
})


In [54]:
test_clean=test.drop(columns=['ID'])

In [55]:
# Apply one hot encoding
# 2. Fit transform ONLY on the categorical training columns
test_encoded_array = ohe.transform(test_clean[cat_cols])

# 3. Get the new feature names generated by OHE
encoded_feature_names = ohe.get_feature_names_out(cat_cols)

# 4. Convert the NumPy array back into a pandas DataFrame
test_encoded = pd.DataFrame(
    test_encoded_array,
    columns=encoded_feature_names,
    index=test_clean.index
)

test_encoded.head()

,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


In [56]:
test_ohe = pd.concat(
    [test_clean.drop(columns=cat_cols), test_encoded],
    axis=1
)

test_ohe.head()

,arpu,age,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_volume,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_highest_amount,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_highest_amount,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_highest_amount,m3_received_senders,m4_received_volume,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_volume,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_volume,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_highest_amount,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_highest_amount,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_highest_amount,m3_deposit_agents,m4_deposit_volume,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_volume,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_volume,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_highest_amount,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_highest_amount,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_highest_amount,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_highest_amount,m4_withdraw_agents,m5_withdraw_volume,m5_withdraw_total_value,m5_withdraw_highest_amount,m5_withdraw_agents,m6_withdraw_volume,m

In [57]:
## Adjust the number of features
test_ohe_adj=test_ohe.drop(columns=features_to_drop)
test_ohe_adj.head()

,age,x_90_d_activity_rate,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_total_value,m2_merchantpay_merchants,m3_merchantpay_total_value,m3_merchantpay_merchants,m4_merchantpay_total_value,m4_merchantpay_merchants,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_total_value,m6_merchantpay_merchants,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_banks,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_recipients,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_recipients,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_senders,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_agents,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_agents,m5_withdraw_total_value,m5_withdraw_highest_amount,m5_withdraw_agents,m6_withdraw_volume,m6_withdraw_total_value,m6_withdraw_agents,m1_daily_avg_bal,m3_daily_avg_bal,m4_daily_avg_bal,m5_daily_avg_bal,m6_daily_avg_bal,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
0,47,0.6905,0.00,0.00,0.0,0.00,0.00,0.0,21341.77,5938.05,8.0,7831.85,1526.06,3.0,28295.28,6259.96,3.0,0.0,0.0,0.0,1982.82,2.0,1505.08,11.0,4373.46,4.0,3790.56,5.0,4346.68,9.0,23484.09,11.0,27009.99,11265.21,11.0,0.00,0.00,0.0,4,42572.01,8.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.00,0.0,3,11931.62,6.0,37721.15,3275.68,20.0,0,0.00,0.0,25476.62,3207.89,18.0,0.00,0.00,0.0,9137.70,4157.57,1.0,20,65202.88,2.0,13,76219.65,20.0,0,0.00,0.0,0.00,0.00,0.0,49904.10,2781.92,13.0,48561.73,2258.02,4.0,14,109128.91,1.0,13,120493.24,1.0,10,111806.67,3.0,0.00,0.00,0.0,189727.10,4672.05,5.0,262377.52,17235.39,2.0,9,40811.44,5.0,11,62921.82,3.0,0,0.00,0.0,0,0.00,0.0,79416.13,4663.77,3.0,0,0.00,0.0,7272.55,17239.77,12989.74,11233.97,16461.21,0.0,0.0,0

In [59]:
test_prob=base_model.predict_proba(test_ohe_adj)[:, 1]

In [60]:
submission[SUBMISSION_TARGET] = test_prob

In [61]:
submission

,ID,Target
0,ID_ED91AC7A28,0.032940
1,ID_29975A9C7E,0.024538
2,ID_F59CE25650,0.035332
3,ID_E875F68D1A,0.016300
4,ID_2902B1C0D0,0.007438
...,...,...
29995,ID_3C7CF79AEE,0.031750
29996,ID_F1651E73C4,0.032826
29997,ID_FD3F0934CA,0.149084
29998,ID_DA6916D6C7,0.022693


just a quick inspection

In [62]:
classification = (test_prob >= 0.5).astype(int)

In [63]:
classification

array([0, 0, 0, ..., 0, 0, 0], shape=(30000,))

In [64]:
submission2=submission.copy()

In [65]:
submission2['classification']=classification
submission2

,ID,Target,classification
0,ID_ED91AC7A28,0.032940,0
1,ID_29975A9C7E,0.024538,0
2,ID_F59CE25650,0.035332,0
3,ID_E875F68D1A,0.016300,0
4,ID_2902B1C0D0,0.007438,0
...,...,...,...
29995,ID_3C7CF79AEE,0.031750,0
29996,ID_F1651E73C4,0.032826,0
29997,ID_FD3F0934CA,0.149084,0
29998,ID_DA6916D6C7,0.022693,0


In [66]:
submission2['classification'].value_counts()

classification
0    27931
1     2069
Name: count, dtype: int64

In [67]:
# Save to CSV
submission.to_csv("submission_LGBM_mined.csv", index=False)
print("Submission file saved!")


Submission file saved!


lets try submitting the oversampled version